In this code, I will look to merge all of the data files so that we have one dataset in which each row corresponds to a single individual, their demographic information, their health information, and their nutritional information. I will also construct an intermediate table, which will remain unaggregated with the goal of possibly including more complex predictors from additional sites later on.

Imports

In [1]:
import pandas as pd
from pathlib import Path

We begin by reading in all of our data files:

In [3]:
raw = Path("data/raw/2017_2018")

# Read function
def read(path):
    # .xpt
    if path.suffix.lower() == ".xpt":
        df = pd.read_sas(path, format="xport", encoding="utf-8")
    
    # sas
    elif path.suffix.lower() == ".sas7bdat":
        df = pd.read_sas(path, encoding="utf-8")
    
    # excel
    else:
        df = pd.read_excel(path)
    
    # Normalize to capitals
    df.columns = df.columns.str.upper()

    # Cast SEQN to integer (from float)
    if "SEQN" in df.columns:
        df["SEQN"] = df["SEQN"].astype(int)
    return df

# NHANES files
nhanes_names = ["DEMO", "DR1IFF", "DR2IFF", "DR1TOT", "DRXFCD", "GHB", "HDL", "TCHOL",
                "HSCRP", "BMX", "BPX", "DIQ", "SMQ", "PAQ", "FSQ"]

# NHANES dictionary, where file name maps to dataframe
# Each glob will only return one object, so we just take the one object

data = {}

for name in nhanes_names:
    print(name)
    data[name] = read(next(raw.glob(f"{name}_J.*")))

# USDA files
usda_files = {
    "FPED_DB":      "FPED_1718.xls",
    "FLAV_VAL":     "flavval.sas7bdat",
    "FLAV_DESC":    "flavdesc.sas7bdat",
    "FLAV_FOODS":   "mainfooddesc.sas7bdat",
    "FPED_TOT1":    "fped_dr1tot_1718.sas7bdat",
    "FPED_TOT2":    "fped_dr2tot_1718.sas7bdat",
    "FLAV_TOT1":    "flav_dr1tot_1718.sas7bdat",
    "FLAV_TOT2":    "flav_dr2tot_1718.sas7bdat",
}

# More of the same, it only looks different because we are adding to a pre-existing dictionary
for name, file in usda_files.items():
    data[name] = read(raw / file)

# Inventory Summary
for name, df in data.items():
    people = df["SEQN"].nunique() if "SEQN" in df.columns else "-"
    print(f"{name:11} rows={len(df):>8,}  people={people:>6}  cols={df.shape[1]}")

DEMO
DR1IFF
DR2IFF
DR1TOT
DRXFCD
GHB
HDL
TCHOL
HSCRP
BMX
BPX
DIQ
SMQ
PAQ
FSQ
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
DEMO        rows=   9,254  people=  9254  cols=46
DR1IFF      rows= 112,683  people=  7640  cols=84
DR2IFF      rows=  93,500  people=  6639  cols=84
DR1TOT      rows=   8,704  people=  8704  cols=168
DRXFCD      rows=   7,083  people=     -  cols=3
GHB         rows=   6,401  people=  6401  cols=2
HDL         rows=   7,435  people=  7435  cols=3
TCHOL       rows=   7,435  people=  7435  cols=3
HSCRP       rows=   8,366  people=  8366  cols=3
BMX         rows=   8,704  people=  8704  cols=21
BPX         rows=   8,704  people=  8704  cols=21
DIQ         rows=   8,897  people=  8897  cols=54
SMQ         rows=   6,724  people=  6724  cols=37
PAQ         rows=   5,856  people=  5856  cols=17
FSQ         rows=   9,254  people=  9254  cols=49
FPED_DB     rows=   7,083  people=     -  cols=39
FLAV_VAL    rows= 262,071  people=     -  cols=5
FLAV_